In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ==================== Configuration ====================
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

RESULTS_DIR = './results'
FIGURES_DIR = './figures'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# Paths to input data
CLINICAL_PATH = './data/clinical.csv'
PATHO_PATH = './data/pathological.csv'
RECURRENCE_PATH = './data/recurrence.csv'
RADIOMICS_ICC_PATH = './results/radiomics.csv'                # ICC + correlation filtered
RADIOMICS_DISTINCTIVE_PATH = './results/radiomics_distinctive.csv'  # GMM selected (all significant)

# Settings for cross-validation
N_SPLITS = 5
N_REPEATS = 10
RANDOM_STATE_CV = 42

# Feature combinations to test
COMBINATIONS = ['Clinical+Radiomics', 'Clinical+Pathological+Radiomics']

# Number of top features to test in experiment 2
STEP = 20
TOP_FEATURE_RANGE = list(range(10, 413, STEP))
if TOP_FEATURE_RANGE[-1] < 412:
    TOP_FEATURE_RANGE.append(412)
TOP_FEATURE_RANGE = sorted(set(TOP_FEATURE_RANGE))

# Set global plotting style to match step3
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14
})

# ==================== Helper functions ====================
def load_and_align_data():
    """Load all CSV files and align by PatientID."""
    clinical = pd.read_csv(CLINICAL_PATH, index_col=0)
    patho = pd.read_csv(PATHO_PATH, index_col=0)
    recurrence = pd.read_csv(RECURRENCE_PATH, index_col=0)
    radiomics_icc = pd.read_csv(RADIOMICS_ICC_PATH, index_col=0)

    # Find common PatientIDs across all data sources
    common_ids = set(clinical.index) & set(patho.index) & set(recurrence.index) & set(radiomics_icc.index)
    common_ids = sorted(common_ids)
    print(f"Common patients: {len(common_ids)}")

    clinical = clinical.loc[common_ids]
    patho = patho.loc[common_ids]
    recurrence = recurrence.loc[common_ids]
    radiomics_icc = radiomics_icc.loc[common_ids]

    # Target
    y = recurrence.iloc[:, 0].values
    print(f"Target distribution: {pd.Series(y).value_counts().to_dict()}")

    # Simple imputation for missing values (median for numerical, mode for categorical)
    def impute_df(df):
        for col in df.columns:
            if df[col].dtype == 'object' or df[col].nunique() < 6:
                df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 0, inplace=True)
            else:
                df[col].fillna(df[col].median(), inplace=True)
        return df

    clinical = impute_df(clinical)
    patho = impute_df(patho)

    # Load distinctive features list (from step2)
    radiomics_dist = pd.read_csv(RADIOMICS_DISTINCTIVE_PATH)
    if 'Feature' not in radiomics_dist.columns:
        radiomics_dist = radiomics_dist.reset_index()
        radiomics_dist.rename(columns={radiomics_dist.columns[0]: 'Feature'}, inplace=True)
    if 'Importance' in radiomics_dist.columns:
        radiomics_dist = radiomics_dist.sort_values('Importance', ascending=False)
    top_features_all = radiomics_dist['Feature'].tolist()
    top_features_all = [f for f in top_features_all if f in radiomics_icc.columns]
    print(f"Distinctive features available in radiomics_icc: {len(top_features_all)}")

    return clinical, patho, radiomics_icc, top_features_all, y

def prepare_feature_set(clinical, patho, radiomics_features, combo_name):
    """Combine clinical, pathological and radiomics features (inner join on index)."""
    if combo_name == 'Clinical+Radiomics':
        X = clinical.join(radiomics_features, how='inner')
    elif combo_name == 'Clinical+Pathological+Radiomics':
        X = clinical.join(patho, how='inner').join(radiomics_features, how='inner')
    else:
        raise ValueError(f"Unknown combination: {combo_name}")
    return X

def evaluate_model(X, y, model, cv):
    """Perform cross-validation and return AUC scores."""
    pipeline = make_pipeline(StandardScaler(), model)
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    return scores

def star_from_pvalue(p):
    """Return significance stars based on p-value."""
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return 'n.s.'

# ==================== Experiment 1: Compare pipelines ====================
def run_experiment1(clinical, patho, radiomics_icc, top_features_all, y):
    print("\n" + "="*60)
    print("Experiment 1: GMM-filtered vs. conventional ICC+corr+LASSO pipeline")
    print("="*60)

    n_top_paper = min(50, len(top_features_all))
    top_50 = top_features_all[:n_top_paper]
    radiomics_gmm = radiomics_icc[top_50]

    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE_CV)

    # Models: GMM-filtered uses standard L2 logistic regression (no additional feature selection)
    # Conventional uses L1 logistic regression (LASSO) for embedded feature selection
    model_gmm = LogisticRegression(penalty='l2', C=1.0, random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced')
    model_lasso = LogisticRegression(penalty='l1', solver='liblinear', C=1.0, random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced')

    # Store scores and compute p-values
    scores_dict = {}
    p_values = {}

    for combo in COMBINATIONS:
        X_gmm = prepare_feature_set(clinical, patho, radiomics_gmm, combo)
        scores_gmm = evaluate_model(X_gmm, y, model_gmm, cv)
        scores_dict[f"{combo}_GMM-filtered"] = scores_gmm

        X_conv = prepare_feature_set(clinical, patho, radiomics_icc, combo)  # all ICC-filtered features
        scores_conv = evaluate_model(X_conv, y, model_lasso, cv)
        scores_dict[f"{combo}_LASSO"] = scores_conv

        t_stat, p_val = stats.ttest_rel(scores_gmm, scores_conv)
        p_values[combo] = p_val

    # Print Experiment 1 main results
    print("\n--- Experiment 1 Main Results ---")
    for combo in COMBINATIONS:
        gmm_key = f"{combo}_GMM-filtered"
        lasso_key = f"{combo}_LASSO"
        gmm_mean = np.mean(scores_dict[gmm_key])
        gmm_std = np.std(scores_dict[gmm_key])
        lasso_mean = np.mean(scores_dict[lasso_key])
        lasso_std = np.std(scores_dict[lasso_key])
        p_val = p_values[combo]
        sig_star = star_from_pvalue(p_val)
        
        print(f"\nFeature Combination: {combo}")
        print(f"  GMM-filtered AUC: {gmm_mean:.4f} ± {gmm_std:.4f}")
        print(f"  LASSO AUC: {lasso_mean:.4f} ± {lasso_std:.4f}")
        print(f"  Paired t-test p-value: {p_val:.6f} ({sig_star})")

    # Plotting: two subplots side by side
    fig, axes = plt.subplots(1, 2, figsize=(10, 8), sharey=True)
    fig.subplots_adjust(top=0.85)
    fig.suptitle('Performance Distribution\nGMM-filtered vs. Conventional Feature Selection', fontsize=20)

    # Colors for methods
    method_colors = {'GMM-filtered': 'lightblue', 'LASSO': 'lightgreen'}

    # Determine y-axis limits with extra space for significance markers
    all_scores = np.concatenate([scores_dict[key] for key in scores_dict.keys()])
    y_min, y_max = np.min(all_scores), np.max(all_scores)
    y_range = y_max - y_min
    y_max_extended = y_max + 0.15 * y_range

    for idx, combo in enumerate(COMBINATIONS):
        ax = axes[idx]
        gmm_key = f"{combo}_GMM-filtered"
        lasso_key = f"{combo}_LASSO"
        data = [scores_dict[gmm_key], scores_dict[lasso_key]]
        labels = ['GMM-filtered', 'LASSO']

        # Boxplot
        bp = ax.boxplot(data, labels=labels, patch_artist=True,
                        boxprops=dict(linewidth=1.5),
                        medianprops=dict(color='black', linewidth=1.5),
                        whiskerprops=dict(linewidth=1.5),
                        capprops=dict(linewidth=1.5))
        for patch, color in zip(bp['boxes'], [method_colors['GMM-filtered'], method_colors['LASSO']]):
            patch.set_facecolor(color)

        # Add significance annotation
        p_val = p_values[combo]
        stars = star_from_pvalue(p_val)
        x1, x2 = 1, 2
        y_marker = y_max + 0.05 * y_range
        ax.plot([x1, x2], [y_marker, y_marker], color='black', linewidth=1.5)
        ax.text((x1+x2)/2, y_marker + 0.01*y_range, stars, ha='center', va='bottom',
                fontsize=16, fontweight='bold')

        ax.set_title(combo, fontsize=16)
        ax.set_ylabel('AUC' if idx == 0 else '')
        ax.grid(axis='y', alpha=0.3)
        ax.set_ylim(y_min, y_max_extended)

    plt.savefig(os.path.join(FIGURES_DIR, 'exp1_comparison.png'), dpi=300)
    plt.close()

    # Save results
    df_res = pd.DataFrame(scores_dict)
    df_res.to_csv(os.path.join(RESULTS_DIR, 'exp1_results.csv'), index=False)
    print("\nExperiment 1 results saved.")

# ==================== Experiment 2: Vary number of top features ====================
def run_experiment2(clinical, patho, radiomics_icc, top_features_all, y):
    print("\n" + "="*60)
    print("Experiment 2: Impact of number of top distinctive features")
    print("="*60)

    max_feat = len(top_features_all)
    if max_feat < 10:
        print("Warning: Not enough distinctive features, skipping experiment 2.")
        return

    feat_range = [k for k in TOP_FEATURE_RANGE if k <= max_feat]
    if max_feat not in feat_range:
        feat_range.append(max_feat)
    feat_range = sorted(feat_range)

    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE_CV)
    # Use standard L2 logistic regression for all models after GMM filtering
    model = LogisticRegression(penalty='l2', C=1.0, random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced')

    # Colors for combinations
    combo_colors = {
        'Clinical+Radiomics': ('blue', 'lightblue'),
        'Clinical+Pathological+Radiomics': ('red', 'lightcoral')
    }

    results = {combo: {'n_features': [], 'auc_mean': [], 'auc_std': []} for combo in COMBINATIONS}
    opt_points = {}

    for combo in COMBINATIONS:
        print(f"\n--- Combination: {combo} ---")
        for k in feat_range:
            top_k = top_features_all[:k]
            radio_k = radiomics_icc[top_k]
            X = prepare_feature_set(clinical, patho, radio_k, combo)
            scores = evaluate_model(X, y, model, cv)
            results[combo]['n_features'].append(k)
            results[combo]['auc_mean'].append(np.mean(scores))
            results[combo]['auc_std'].append(np.std(scores))
            print(f"k={k:3d}: AUC = {np.mean(scores):.3f} ± {np.std(scores):.3f}")

        # Find optimal
        idx_opt = np.argmax(results[combo]['auc_mean'])
        opt_k = results[combo]['n_features'][idx_opt]
        opt_auc = results[combo]['auc_mean'][idx_opt]
        opt_points[combo] = (opt_k, opt_auc)

    # Plotting
    plt.figure(figsize=(10, 8))
    for combo in COMBINATIONS:
        color_main, color_fill = combo_colors[combo]
        x = results[combo]['n_features']
        y_mean = results[combo]['auc_mean']
        y_std = results[combo]['auc_std']

        # Mean line
        plt.plot(x, y_mean, color=color_main, linewidth=2, label=combo)
        # Confidence band (mean ± std)
        plt.fill_between(x, np.array(y_mean) - np.array(y_std),
                         np.array(y_mean) + np.array(y_std),
                         color=color_fill, alpha=0.5)
        # Optimal point (hollow circle)
        opt_k, opt_auc = opt_points[combo]
        plt.plot(opt_k, opt_auc, marker='o', markersize=10,
                 markerfacecolor='none', markeredgecolor=color_main,
                 markeredgewidth=2)

    plt.axvline(x=50, color='gray', linestyle='--', alpha=0.7, label='GMM-filtered default (k=50)')
    plt.xlabel('Number of top distinctive radiomics features')
    plt.ylabel('AUC')
    plt.title('Performance Distribution\nDifferent numbers of GMM-Selected Features', fontsize=18)
    plt.legend(loc='best')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'exp2_feature_number.png'), dpi=300)
    plt.close()

    # Save results
    for combo in COMBINATIONS:
        df = pd.DataFrame(results[combo])
        df.to_csv(os.path.join(RESULTS_DIR, f'exp2_{combo.replace("+","_")}.csv'), index=False)
    print("\nExperiment 2 results saved.")
    print("Optimal number of features:")
    for combo, (opt_k, opt_auc) in opt_points.items():
        print(f"  {combo}: {opt_k} features (AUC = {opt_auc:.3f})")

# ==================== Main ====================
def main():
    print("Loading and preprocessing data...")
    clinical, patho, radiomics_icc, top_features_all, y = load_and_align_data()

    run_experiment1(clinical, patho, radiomics_icc, top_features_all, y)
    run_experiment2(clinical, patho, radiomics_icc, top_features_all, y)

    print("\nAll experiments completed. Results and figures saved.")

if __name__ == "__main__":
    main()

Loading and preprocessing data...
Common patients: 142
Target distribution: {1: 86, 0: 56}
Distinctive features available in radiomics_icc: 412

Experiment 1: GMM-filtered vs. conventional ICC+corr+LASSO pipeline

--- Experiment 1 Main Results ---

Feature Combination: Clinical+Radiomics
  GMM-filtered AUC: 0.7887 ± 0.0751
  LASSO AUC: 0.7621 ± 0.0581
  Paired t-test p-value: 0.026316 (*)

Feature Combination: Clinical+Pathological+Radiomics
  GMM-filtered AUC: 0.8778 ± 0.0579
  LASSO AUC: 0.8251 ± 0.0549
  Paired t-test p-value: 0.000003 (***)

Experiment 1 results saved.

Experiment 2: Impact of number of top distinctive features

--- Combination: Clinical+Radiomics ---
k= 10: AUC = 0.750 ± 0.080
k= 30: AUC = 0.784 ± 0.068
k= 50: AUC = 0.789 ± 0.075
k= 70: AUC = 0.782 ± 0.075
k= 90: AUC = 0.761 ± 0.078
k=110: AUC = 0.750 ± 0.081
k=130: AUC = 0.761 ± 0.075
k=150: AUC = 0.764 ± 0.070
k=170: AUC = 0.767 ± 0.075
k=190: AUC = 0.778 ± 0.069
k=210: AUC = 0.776 ± 0.068
k=230: AUC = 0.773 ± 0